In [ ]:
import torch
import mmcv
import mmengine
import mmdet

print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU 사용 가능:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("MMCV:", mmcv.__version__)
print("MMEngine:", mmengine.__version__)
print("MMDetection:", mmdet.__version__)

In [ ]:
from pathlib import Path

ROOT = Path.cwd()

DATASET_ROOT = (
    ROOT / "../../dataset/Circuit testing 2 -test.v1i.coco"
).resolve()

CONFIG_DIR = (ROOT / "configs").resolve()
WORK_DIR = (ROOT / "outputs").resolve()

CONFIG_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

print("현재 위치 :", ROOT)
print("Dataset   :", DATASET_ROOT)
print("존재 여부 :", DATASET_ROOT.exists())

In [ ]:
for split in ["train", "valid", "test"]:
    split_dir = DATASET_ROOT / split
    annotation = split_dir / "_annotations.coco.json"

    images = (
        list(split_dir.glob("*.jpg"))
        + list(split_dir.glob("*.jpeg"))
        + list(split_dir.glob("*.png"))
    )

    print(f"[{split}]")
    print("images     :", len(images))
    print("annotation :", annotation.exists())
    print()

In [ ]:
import json

annotation_path = DATASET_ROOT / "train" / "_annotations.coco.json"

with open(annotation_path, "r", encoding="utf-8") as f:
    coco = json.load(f)
# id=0은 Roboflow 프로젝트용 category이므로 제외
categories = sorted(
    [c for c in coco["categories"] if c["id"] != 0],
    key=lambda x: x["id"]
)

CLASSES = tuple(
    category["name"]
    for category in categories
)

NUM_CLASSES = len(CLASSES)

print("CLASSES =", CLASSES)
print("NUM_CLASSES =", NUM_CLASSES)

category_ids = sorted(
    set(ann["category_id"] for ann in coco["annotations"])
)

print("실제 annotation category IDs:", category_ids)


이 위에까지 쉘 4임


In [ ]:
CLASSES = tuple(
    category["name"]
    for category in categories
)

NUM_CLASSES = len(CLASSES)

print("\nCLASSES =", CLASSES)
print("NUM_CLASSES =", NUM_CLASSES)

In [ ]:
print("이미지 수 :", len(coco["images"]))
print("Annotation 수 :", len(coco["annotations"]))
print("Class 수 :", len(coco["categories"]))

In [ ]:
from collections import Counter

counts = Counter(
    ann["category_id"]
    for ann in coco["annotations"]
)

print("=== 클래스별 객체 수 ===")

for category in categories:  # 이미 id=0 제외한 categories
    category_id = category["id"]
    name = category["name"]

    print(f"{name:20s} : {counts[category_id]}")

In [ ]:
import subprocess
import sys

subprocess.run([
    sys.executable,
    "-m", "mim",
    "download", "mmdet",
    "--config", "rtmdet_m_8xb32-300e_coco",
    "--dest", str(CONFIG_DIR)
], check=True)

In [ ]:
for file in CONFIG_DIR.iterdir():
    print(file.name)

In [ ]:
from mmengine.config import Config

BASE_CONFIG = CONFIG_DIR / "rtmdet_m_8xb32-300e_coco.py"

cfg = Config.fromfile(str(BASE_CONFIG))

print("Config 로드 성공")
print("기존 클래스 수:", cfg.model.bbox_head.num_classes)

In [ ]:
# COCO 80 classes → PCB 8 classes
cfg.model.bbox_head.num_classes = NUM_CLASSES

metainfo = {
    "classes": CLASSES
}

# =========================
# Train
# =========================
train_dataset = cfg.train_dataloader.dataset

train_dataset.data_root = str(DATASET_ROOT) + "/"
train_dataset.ann_file = "train/_annotations.coco.json"
train_dataset.data_prefix = {
    "img": "train/"
}
train_dataset.metainfo = metainfo


# =========================
# Validation
# =========================
val_dataset = cfg.val_dataloader.dataset

val_dataset.data_root = str(DATASET_ROOT) + "/"
val_dataset.ann_file = "valid/_annotations.coco.json"
val_dataset.data_prefix = {
    "img": "valid/"
}
val_dataset.metainfo = metainfo


# =========================
# Test
# =========================
test_dataset = cfg.test_dataloader.dataset

test_dataset.data_root = str(DATASET_ROOT) + "/"
test_dataset.ann_file = "test/_annotations.coco.json"
test_dataset.data_prefix = {
    "img": "test/"
}
test_dataset.metainfo = metainfo


# =========================
# Evaluator
# =========================
cfg.val_evaluator.ann_file = str(
    DATASET_ROOT / "valid" / "_annotations.coco.json"
)

cfg.test_evaluator.ann_file = str(
    DATASET_ROOT / "test" / "_annotations.coco.json"
)


# 확인
print("클래스 수 :", cfg.model.bbox_head.num_classes)
print("Classes   :", CLASSES)
print("Train     :", train_dataset.ann_file)
print("Valid     :", val_dataset.ann_file)
print("Test      :", test_dataset.ann_file)

In [ ]:
cfg.train_dataloader.batch_size = 2
cfg.val_dataloader.batch_size = 1
cfg.test_dataloader.batch_size = 1

cfg.train_dataloader.num_workers = 2
cfg.val_dataloader.num_workers = 2
cfg.test_dataloader.num_workers = 2

cfg.train_cfg.max_epochs = 100
cfg.train_cfg.val_interval = 5

cfg.work_dir = str(WORK_DIR)

print("Batch :", cfg.train_dataloader.batch_size)
print("Epoch :", cfg.train_cfg.max_epochs)
print("Output:", cfg.work_dir)

In [ ]:
print("=== 변경 전 ===")
for transform in train_dataset.pipeline:
    print(transform["type"])

train_dataset.pipeline = [
    transform
    for transform in train_dataset.pipeline
    if transform["type"] != "YOLOXHSVRandomAug"
]

print("\n=== 변경 후 ===")
for transform in train_dataset.pipeline:
    print(transform["type"])

In [ ]:
print("=== train_cfg ===")
print(cfg.train_cfg)

print("\n=== param_scheduler ===")
print(cfg.param_scheduler)

print("\n=== load_from ===")
print(cfg.get("load_from", None))

In [ ]:
# =========================
# 100 Epoch용 학습 스케줄
# =========================

cfg.train_cfg.max_epochs = 100
cfg.train_cfg.val_interval = 5

# 마지막 구간 설정
cfg.train_cfg.dynamic_intervals = [
    (90, 1)
]

# Warm-up
cfg.param_scheduler[0].begin = 0
cfg.param_scheduler[0].end = 500

# Cosine Annealing
cfg.param_scheduler[1].begin = 50
cfg.param_scheduler[1].end = 100
cfg.param_scheduler[1].T_max = 50

print("=== 변경 후 train_cfg ===")
print(cfg.train_cfg)

print("\n=== 변경 후 scheduler ===")
print(cfg.param_scheduler)

In [ ]:
checkpoints = list(CONFIG_DIR.glob("rtmdet_m*.pth"))

if len(checkpoints) == 0:
    raise FileNotFoundError("RTMDet-m pretrained checkpoint를 찾을 수 없습니다.")

PRETRAINED = checkpoints[0]

cfg.load_from = str(PRETRAINED)

print("Pretrained weight:")
print(cfg.load_from)

In [ ]:
CUSTOM_CONFIG = CONFIG_DIR / "rtmdet_m_pcb.py"

cfg.dump(str(CUSTOM_CONFIG))

print("Config 저장 완료")
print(CUSTOM_CONFIG)

In [ ]:
print("=== 최종 학습 설정 ===")
print("Classes       :", CLASSES)
print("Num Classes   :", cfg.model.bbox_head.num_classes)
print("Batch Size    :", cfg.train_dataloader.batch_size)
print("Max Epochs    :", cfg.train_cfg.max_epochs)
print("Val Interval  :", cfg.train_cfg.val_interval)
print("Dataset       :", DATASET_ROOT)
print("Work Dir      :", cfg.work_dir)
print("Pretrained    :", cfg.load_from)

print("\nTrain pipeline:")
for transform in cfg.train_dataloader.dataset.pipeline:
    print("-", transform["type"])

In [ ]:
from mmengine.runner import Runner

runner = Runner.from_cfg(cfg)
runner.train()